# Distran: bounded Amazon overlap audit
Upload two locally obtained JSONL review samples to Colab after reviewing the original dataset terms at https://amazon-reviews-2023.github.io/. Name them Video_Games.jsonl and Movies_and_TV.jsonl. No automatic multi-gigabyte downloads. This bounded prefix audit is biased and is not an evaluation dataset. A zero-overlap result means do not proceed with cross-domain training from this sample.

In [ ]:
from pathlib import Path
import json, hashlib
from collections import Counter
LIMIT=100_000
paths={d:Path(d+".jsonl") for d in ["Video_Games","Movies_and_TV"]}
missing=[str(p) for p in paths.values() if not p.exists()]
assert not missing, "Upload source files first: " + ", ".join(missing)

In [ ]:
rows=[]; users={}; manifest={}
for domain,path in paths.items():
    ids=set(); count=0; digest=hashlib.sha256()
    with path.open("rb") as f:
        for n,line in enumerate(f):
            if n>=LIMIT: break
            digest.update(line)
            r=json.loads(line)
            if not all(k in r for k in ["user_id","parent_asin","rating","timestamp"]): continue
            ids.add(r["user_id"]); count+=1
            rows.append({"user_id":r["user_id"],"item_id":domain+":"+r["parent_asin"],"rating":float(r["rating"]),"timestamp":int(r["timestamp"]),"domain":domain})
    users[domain]=ids
    manifest[domain]={"valid_rows":count,"unique_users":len(ids),"consumed_prefix_sha256":digest.hexdigest(),"sample":"first 100000 lines maximum"}
overlap=set.intersection(*users.values())
print(json.dumps(manifest,indent=2)); print("Overlapping source users:",len(overlap))

Before training: deduplicate (user, domain, parent_asin, timestamp); retain source rating meaning; audit editions and suspicious activity. Use globally chronological train/validation/test cutoffs. Calculate overlap in TRAIN only for a warm cross-domain cohort; evaluate cold users/items separately. Fit all mappings and metadata features on train-time data. Do not identify these users with Distran accounts. Export only pseudonymized research IDs outside this runtime. See the baseline notebook for BPR and the README for full evaluation requirements.